In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Import 

In [2]:
from models import Simple_Model
from utils import EarlyStopping,EpochTrainer, get_device

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler

# Parameters

In [19]:
batch_size       = 64
epochs           = 100
hidden_size      = 128
num_output       = 1
num_hidden_layer = 3
dropout          = 0.3
patience         = 10
test_size        = 0.2
lr = 0.001

# Get data

## Option 1 : Get raw data (random)

In [4]:
features_num = 10
record_cnt = 100

X = torch.randint( 0, 10, (record_cnt, features_num))
y = torch.randint( 0, 10, (record_cnt,))

## Option 2 :  Get raw data from sklearn dataset

In [5]:
# X,y = load_diabetes(return_X_y = True)

# Data Processing

## split data into training and testing dataset

In [6]:
y = y.reshape( len(y), 1)

In [7]:
X_train , X_test , y_train, y_test = train_test_split(X, y, test_size = test_size)

## Data scaling

In [8]:
X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

preprocess_config = {
    "x_mean": X_scaler.mean_.tolist(),
    "x_scale": X_scaler.scale_.tolist(),
    "y_mean": y_scaler.mean_.tolist(),
    "y_scale": y_scaler.scale_.tolist(),
    "task_type": "regression",
}

## Transform to Tensor dataset

In [9]:
X_train_tensor = torch.from_numpy( X_train_scaled).float()
X_test_tensor = torch.from_numpy( X_test_scaled).float()

y_train_tensor = torch.from_numpy( y_train_scaled).float()
y_test_tensor = torch.from_numpy( y_test_scaled).float()

In [10]:
train_dataset = TensorDataset( X_train_tensor, y_train_tensor)
test_datset = TensorDataset( X_test_tensor, y_test_tensor)

In [11]:
train_loader = DataLoader( train_dataset, batch_size = batch_size, shuffle = True )
test_loader = DataLoader( test_datset, batch_size = batch_size, shuffle = True )

# Prepare model

In [12]:
device = get_device()

In [13]:
num_cols = X_train.shape[1]

In [14]:
model = Simple_Model(
    input_size = num_cols,
    hidden_size = hidden_size,
    num_output = num_output,
    num_hidden_layer = num_hidden_layer,
    dropout = dropout 
 )

_ = model.to(device)

In [15]:
optimizer = torch.optim.Adam(model.parameters(), lr = lr)
criterion = nn.MSELoss()

model_config = {
    "input_size": num_cols,
    "hidden_size": hidden_size,
    "num_output": num_output,
    "num_hidden_layer": num_hidden_layer,
    "dropout": dropout,
}

In [16]:
early_stopping = EarlyStopping(
    patience = patience,
    path='../checkpoints/simple_checkpoint.pt',
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
    },
)

# Loop epochs and train model

In [17]:
epoch_trainer = EpochTrainer(
    model = model,
    early_stopping = early_stopping,
    device = device,
    optimizer = optimizer,
    criterion = criterion,
    eval_method = 'R2'
)

In [20]:
for epoch in range(epochs):

    avg_train_loss, avg_val_loss, result = epoch_trainer(train_loader , test_loader )

    for key, value in result.items():        
        print(f"Epoch {epoch + 1:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | {key}: {value:.4f}")
    
    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        
        print("Early stopping triggered! Training stopped.")
        
        break
        

Epoch   1 | Train Loss: 0.9009 | Val Loss: 0.9902 | r2: -0.0652
Epoch   2 | Train Loss: 1.0645 | Val Loss: 0.9826 | r2: -0.0570
Epoch   3 | Train Loss: 0.9889 | Val Loss: 0.9770 | r2: -0.0510
Epoch   4 | Train Loss: 1.0628 | Val Loss: 0.9712 | r2: -0.0448
Epoch   5 | Train Loss: 0.9385 | Val Loss: 0.9668 | r2: -0.0400
Epoch   6 | Train Loss: 1.0382 | Val Loss: 0.9648 | r2: -0.0379
Epoch   7 | Train Loss: 1.1439 | Val Loss: 0.9644 | r2: -0.0375
Epoch   8 | Train Loss: 1.1026 | Val Loss: 0.9645 | r2: -0.0375
Epoch   9 | Train Loss: 0.7754 | Val Loss: 0.9637 | r2: -0.0367
Epoch  10 | Train Loss: 1.1477 | Val Loss: 0.9633 | r2: -0.0363
Epoch  11 | Train Loss: 0.9477 | Val Loss: 0.9648 | r2: -0.0378
Epoch  12 | Train Loss: 0.9600 | Val Loss: 0.9657 | r2: -0.0388
Epoch  13 | Train Loss: 0.8556 | Val Loss: 0.9649 | r2: -0.0380
Epoch  14 | Train Loss: 0.9041 | Val Loss: 0.9622 | r2: -0.0351
Epoch  15 | Train Loss: 0.9554 | Val Loss: 0.9567 | r2: -0.0292
Epoch  16 | Train Loss: 0.9528 | Val Los

In [ ]:
# for epoch in range(epoch):
    
#     # ------------------- Training -------------------
#     model.train()
#     train_loss = 0.0

    
#     for batch_x, batch_y in train_loader:
        
#         batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
#         optimizer.zero_grad()
#         output = model(batch_x)

#         loss = criterion(output.squeeze(), batch_y)
        
#         loss.backward()
#         optimizer.step()

#         train_loss += loss.item()

#     avg_train_loss = train_loss / len(train_loader)

#     # ------------------- Validation -------------------

#     model.eval()
#     val_loss = 0.0

#     all_preds = []
#     all_actual = []

#     with torch.no_grad():
        
#         for batch_x, batch_y in test_loader:

#             batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
#             predit = model(batch_x).squeeze()
#             actual = batch_y.squeeze()


#             loss = criterion(predit, batch_y)
#             val_loss += loss.item()

            
#             all_preds.append(predit)
#             all_actual.append(actual)


#     # ------------------- Evaluation -------------------    
    
#     # Average loss
#     avg_val_loss = val_loss / len(test_loader)          

#     # R2
#     all_preds   = torch.cat(all_preds)
#     all_actual = torch.cat(all_actual)

#     ss_res = ((all_actual - all_preds) ** 2).sum()
#     ss_tot = ((all_actual - all_actual.mean()) ** 2).sum()
#     r2 = (1 - ss_res / ss_tot).item()
            

#     print(f'Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | R²: {r2:.4f}')
    
#     # ==================== Early Stopping Check ====================
#     early_stopping(avg_val_loss, model)
    
#     if early_stopping.early_stop:
#         print("Early stopping triggered! Training stopped.")
#         break

# print("Training finished!")
    